In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -x bin/micromamba ]; then
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/chem-mamba
bin/micromamba create -y -p /content/chem-psi4 -c conda-forge --override-channels python=3.12 psi4=1.11 rdkit matplotlib scipy
/content/chem-psi4/bin/python -c 'import psi4, rdkit; print(psi4.__version__, rdkit.__version__)'

使用同一個甲醇比較四種電荷分配方法。量化計算採作者先前 notebook 的固定幾何。

選擇 Colab 2026.07 的 CPU 執行階段，由上往下執行。安裝格只在目前的雲端工作階段建立獨立環境。

In [ ]:
from pathlib import Path
import os
import subprocess

python = "/content/chem-psi4/bin/python"
def run(script):
    # 子程序只存圖，不沿用 Colab 核心的 inline 顯示後端。
    env = dict(os.environ, MPLBACKEND="Agg")
    result = subprocess.run([python, "-c", script], capture_output=True, text=True, env=env)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    result.check_returncode()

### 計算

摘要會印在下方。完整的 Psi4 輸出保存在 05-04-atomic-charges.log，可以從工作階段的檔案面板開啟。

In [ ]:
run(r'''
import psi4
psi4.core.set_output_file("05-04-atomic-charges.log", False)
psi4.set_num_threads(2)
psi4.set_memory("1 GB")
methanol = psi4.geometry("""
0 1
C -0.660110446961  0.259591312943  0.097933867804
O  0.606292553039 -0.291476687057 -0.121678132196
H -0.861264446961  0.336100312943  1.188582867804
H -0.726113446961  1.266898312943 -0.367691132196
H -1.427376446961 -0.397005687057 -0.361404132196
H  1.252273553039  0.329044312943  0.305550867804
units angstrom
symmetry c1
""")
energy, wfn = psi4.energy("B3LYP/6-31G(d)", molecule=methanol, return_wfn=True)
for method in ["MULLIKEN_CHARGES", "LOWDIN_CHARGES"]:
    psi4.oeprop(wfn, method)
    values = wfn.atomic_point_charges().np.copy()
    assert len(values) == methanol.natom() and abs(sum(values)) < 1e-5
    print(method, values, "sum =", sum(values))
''')

### 以兩種 RDKit 經驗方法重算同一排原子
總和檢查包含所有顯式氫。圖的每組直條和上格的同一列原子對應。

In [ ]:
run(r'''
from rdkit import Chem
from rdkit.Chem import AllChem, rdPartialCharges
import numpy as np

mol = Chem.AddHs(Chem.MolFromSmiles("CO"))
labels = [f"{a.GetSymbol()}{a.GetIdx()}" for a in mol.GetAtoms()]
rdPartialCharges.ComputeGasteigerCharges(mol, throwOnParamFailure=True)
gasteiger = [a.GetDoubleProp("_GasteigerCharge") for a in mol.GetAtoms()]
props = AllChem.MMFFGetMoleculeProperties(mol)
if props is None:
    raise ValueError("MMFF94 無法指派所有原子參數")
mmff = [props.GetMMFFPartialCharge(i) for i in range(mol.GetNumAtoms())]
for name, values in [("Gasteiger", gasteiger), ("MMFF94", mmff)]:
    assert np.isfinite(values).all() and abs(sum(values) - Chem.GetFormalCharge(mol)) < 1e-5
    print(name, dict(zip(labels, values)), "sum =", sum(values))
from pathlib import Path
import matplotlib.pyplot as plt
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figdir.mkdir(parents=True, exist_ok=True)
positions = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(6.5, 3.9))
ax.bar(positions - 0.19, gasteiger, 0.38, label="Gasteiger", color="#287090")
ax.bar(positions + 0.19, mmff, 0.38, label="MMFF94", color="#bf6731")
ax.axhline(0, lw=0.7, color="black")
ax.set(xticks=positions, xticklabels=labels, ylabel="Partial charge (e)", title="Methanol · hydrogen atoms shown explicitly")
ax.legend()
fig.tight_layout()
fig.savefig(figdir / "05-04-methanol-charges.png", dpi=180)
''')

In [ ]:
from IPython.display import Image, display
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figdir / "05-04-methanol-charges.png")))